# PyNAS quick start

This notebook verifies the installed `pynas` package by loading the bundled configuration, creating a tiny NAS population, building a segmentation model, and running CPU inference on synthetic data. Full training still requires the burned-area dataset described in the repository README.

In [1]:
%load_ext autoreload
%autoreload 2

## Environment check

In [2]:
import torch
import pynas

print("PyNAS version:", pynas.__version__)
print("CUDA available:", torch.cuda.is_available())
print("CUDA version:", torch.version.cuda)
print("GPU count:", torch.cuda.device_count())
print("GPU name:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "None")

PyNAS version: 0.1.0
CUDA available: True
CUDA version: 13.0
GPU count: 2
GPU name: NVIDIA A30-24C


In [3]:
  import sys
  sys.path.insert(0, "/home/heo/project/py-q-nas")

## Load packaged configuration

In [4]:
from pynas.core.config import default_config_path, load_default_config

config = load_default_config()
print("Configuration loaded from:", default_config_path())
print("GA population_size:", config.getint("GA", "population_size"))
assert config.has_section("ConvAct")
assert config.has_section("GA")

Configuration loaded from: /home/heo/projects/py-q-nas/src/pynas/core/config.ini
GA population_size: 20


In [5]:
import sys, os
sys.path.insert(0, os.path.abspath(".."))   # notebooks/ -> repo root

from pathlib import Path
from scripts.dataloader import SegmentationDataModule

In [6]:
root_dir = "../data/burnt_area_dataset/burned.zarr/"   
dm = SegmentationDataModule(root_dir, batch_size=8, num_workers=6, transform=None)

dm.setup()
train_loader = dm.train_dataloader()
images, masks = next(iter(train_loader))
print("Image batch shape:", images.shape)
print("Mask batch shape:", masks.shape)
print("Mask dtype:", masks.dtype)
print("Mask unique values:", masks.unique())
print("dm.num_classes:", dm.num_classes)

Image batch shape: torch.Size([8, 7, 256, 256])
Mask batch shape: torch.Size([8, 256, 256])
Mask dtype: torch.int64
Mask unique values: tensor([0, 1, 2, 3])
dm.num_classes: 4


## Launch NAS

In [ ]:
import torch
import pytorch_lightning as pl
from pynas.core.config import default_config_path, load_default_config
from pynas.core.population import Population
from scripts.dataloader import SegmentationDataModule

# ---- Config ----
config = load_default_config()
print("Configuration loaded from:", default_config_path())

seed = config.getint("Computation", "seed")
pl.seed_everything(seed=seed, workers=True)
torch.set_float32_matmul_precision("medium")

max_layers = config.getint("NAS", "max_layers", fallback=7)
n_individuals = config.getint("GA", "population_size")
k_best = config.getint("GA", "k_best")
n_random = config.getint("GA", "n_random")
mating_pool_cutoff = config.getfloat("GA", "mating_pool_cutoff")
mutation_probability = config.getfloat("GA", "mutation_probability")
max_iterations = config.getint("GA", "max_iterations")
epochs = config.getint("GA", "epochs")
batch_size = config.getint("GA", "batch_size")
max_parameters = config.getint("GA", "max_parameters", fallback=5_000_000)

print(dict(config.items("GA")))

# ---- Real dataset ----
root_dir = "../data/burnt_area_dataset/burned.zarr"
dm = SegmentationDataModule(root_dir, batch_size=batch_size, num_workers=6, transform=None)

pop = Population(n_individuals=20, max_layers=7, dm=dm, max_parameters=5_000_000,
                 save_directory="../models_traced")
pop.initial_poll() 

pop.train_generation(task="segmentation", lr=0.001, epochs=config.getint("GA", "epochs"),
    batch_size=config.getint("GA", "batch_size")
)

# Subsequent generations
for gen in range(max_iterations):
    print(f"=== Generation {gen+1}/{max_iterations} ===")
    pop.evolve(
        mating_pool_cutoff=mating_pool_cutoff,
        mutation_probability=mutation_probability,
        k_best=k_best,
        n_random=n_random,
    )
    pop.train_generation(task="segmentation", lr=0.001, epochs=epochs, batch_size=batch_size)

Seed set to 42


Configuration loaded from: /home/heo/projects/py-q-nas/src/pynas/core/config.ini
{'population_size': '20', 'epochs': '4', 'batch_size': '8', 'max_parameters': '5000000', 'max_iterations': '10', 'logs_dir_ga': './logs/GA_logs', 'mating_pool_cutoff': '0.5', 'mutation_probability': '0.2', 'n_random': '1', 'k_best': '1', 'task': 'segmentation', 'training_workers': '1'}


Generating Population: 100%|████████████████████████| 20/20 [02:06<00:00,  6.34s/it]



  DIVERSITY — GEN 0
    unique architectures: 20/20
    depth distribution:   6L:10, 8L:7, 10L:2, 12L:1
    block usage:
      AvgPool            38  (25.7%)
      MaxPool            36  (24.3%)
      MBConvNoRes        20  (13.5%)
      MBConv             17  (11.5%)
      ResNetBlock        11  (7.4%)
      DenseNetBlock       9  (6.1%)
      ConvSE              8  (5.4%)
      ConvAct             5  (3.4%)
      ConvBnAct           4  (2.7%)


##############################################################################
GENERATION 0 — 20 models | task=segmentation | epochs=4 | train=['fp32', 'fp32_ft', 'int8_torch', 'int8_custom'] | test=['fp32', 'fp32_ft', 'int8_torch', 'int8_custom'] | fitness=int8_custom
##############################################################################

GEN 0 | MODEL 0/19 | params=79,875 | arch=Lne4agn1EPM2ELne6agn1EPM2ELme3agn1EPa2ELRr2agn1EPa2ELco04k3s1p1arn1EPa2ELdo06agn1EPa2EE

-------------------------------------------------------------------

GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
💡 Tip: For seamless cloud uploads and versioning, try installing [litmodels](https://pypi.org/project/litmodels/) to enable LitModelCheckpoint, which syncs automatically with the Lightning model registry.
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0,1]

  | Name    | Type               | Params | Mode  | FLOPs
---------------------------------------------------------------
0 | model   | GenericUNetNetwork | 79.9 K | train | 0    
1 | loss_fn | FocalLoss          | 0      | train | 0    
2 | mse     | MeanSquaredError   | 0      | train | 0    
---------------------------------------------------------------
79.9 K    Trainable params
0         Non-trainable params
79.9 K    Total params

...

LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0,1]

  | Name    | Type               | Params | Mode  | FLOPs
---------------------------------------------------------------
0 | model   | GenericUNetNetwork | 12.6 K | train | 0    
1 | loss_fn | FocalLoss          | 0      | train | 0    
2 | mse     | MeanSquaredError   | 0      | train | 0    
---------------------------------------------------------------
12.6 K    Trainable params
0         Non-trainable params
12.6 K    Total params
0.050     Total estimated model params size (MB)
171       Modules in train mode
0         Modules in eval mode
0         Total Flops


Sanity Checking: |                                            | 0/? [00:00<?, ?it/s]

/home/heo/projects/py-q-nas/.venv/lib/python3.12/site-packages/pytorch_lightning/utilities/_pytree.py:21: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.
/home/heo/projects/py-q-nas/.venv/lib/python3.12/site-packages/pytorch_lightning/trainer/connectors/data_connector.py:429: Consider setting `persistent_workers=True` in 'val_dataloader' to speed up the dataloader worker initialization.
/home/heo/projects/py-q-nas/.venv/lib/python3.12/site-packages/pytorch_lightning/trainer/connectors/data_connector.py:429: Consider setting `persistent_workers=True` in 'train_dataloader' to speed up the dataloader worker initialization.


Training: |                                                   | 0/? [00:00<?, ?it/s]

Validation: |                                                 | 0/? [00:00<?, ?it/s]

## Run inference

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = model.to(device).eval()
example_input = torch.randn(1, *dm.input_shape, device=device)

with torch.no_grad():
    output = model(example_input)

print("Output shape:", tuple(output.shape))
assert output.shape == (1, dm.num_classes, dm.input_shape[1], dm.input_shape[2])

## Create and checkpoint an initial population

In [ ]:
pop.initial_poll()
print(pop.df[["Generation", "Params"]])
assert len(pop.population) == pop.n_individuals
assert (save_directory / "src" / "population_0.pkl").exists()
assert (save_directory / "src" / "df_population_0.pkl").exists()

## Cleanup

In [ ]:
workspace.cleanup()
print("PyNAS quick start completed successfully.")